# 深度学习专题 02 · 一张手势照片能教模型几次？

![同一张 V 手势经过平移、变暗和过度裁切后的并列图](../assets/augmentation_views.png)

课堂上拍下一个字母 V。第二天，同学的手偏右了一点，教室也暗了一些。看上图，遮住图注：哪几张还能凭指尖和手掌判断是 V？哪张让你开始犹豫？**先写下你愿意保留的变换，以及你拒绝一张图的理由。**

这节课要交一份增强配方，写清变换、强度和适用条件，再用相同训练预算检验它。沿用[第一课的 CNN](CNNImageRecognition.ipynb) 和[本地手势数据](data/image_recognition/README.md)。安装 PyTorch 后可从本课第一格独立运行。

## 1 · 先决定标签还能不能用

图像增强会从同一张已标注的训练图生成不同视图，并沿用原标签。这隐含一个需要检查的假设：变化之后，定义这个类别的线索依然在图里。轻微转动模拟手腕姿态，平移模拟取景偏差，亮度缩放模拟曝光变化。手指移出画面、过曝使轮廓消失时，继续贴原标签就可能教错模型。镜像是否合适也要结合任务中左右关系的含义判断。

拖动一个参数，其余先保持默认。找到一个你能辨认的极限，再换另一个字母复查。图下的“原始标签”来自原照片，不能用来证明变换合理。

In [ ]:
# 【准备】可从项目根目录或 MachineLearning 目录运行
import matplotlib.pyplot as plt
import torch
import ipywidgets as widgets
from IPython.display import Markdown, display

try:
    from MachineLearning import image_labs as lab
except ModuleNotFoundError as exc:
    if exc.name != "MachineLearning":
        raise
    import image_labs as lab

torch.set_num_threads(min(4, torch.get_num_threads()))
device = lab.training_device()
signs = lab.load_signs()

x_train, y_train = signs["train"]

def inspect_view(letter="V", sample=0, degrees=0, dx=0, dy=0, gain=1.0):
    class_id = lab.LETTERS.index(letter)
    index = torch.where(y_train == class_id)[0][sample]
    original = x_train[index:index+1]
    view = lab.transform_images(original, degrees=degrees, dx=dx, dy=dy, gain=gain)
    fig, axes = plt.subplots(1, 2, figsize=(4.7, 2.4))
    for ax, image, title in zip(axes, (original, view), ("Original", "New view")):
        ax.imshow(image[0, 0], cmap="gray", vmin=0, vmax=1)
        ax.set_title(title)
        ax.axis("off")
    plt.show()
    print("原始标签", letter)

widgets.interact(inspect_view, letter=widgets.Dropdown(options=list(lab.LETTERS), value="V"),
                 sample=(0, 19), degrees=(-60, 60, 2), dx=(-18, 18), dy=(-18, 18), gain=(0.2, 2., 0.1));

## 2 · 把“稍暗一点”写成可检查的函数

像素已缩放到 0–1。把亮度乘以 0.8，原来为 0.5 的像素变成 0.4。先预测：若改乘 1.5，原本为 0.8 和 1.0 的两个像素还能区分吗？下面将越界值截到 1。完成这一小步后，再组合空间变换。

**练习**　改一行代码，把这四个像素调暗 20%，在运行前写下四个答案。然后恢复增亮并观察有几个像素堆到了上限。

In [ ]:
def adjust_light(image, gain):
    return (image * gain).clamp(0, 1)

strip = torch.tensor([0., 0.5, 0.8, 1.0])
my_gain = 1.5
print(adjust_light(strip, my_gain))
torch.testing.assert_close(adjust_light(strip, 0.8), torch.tensor([0., .4, .64, .8]))
torch.testing.assert_close(adjust_light(strip, 1.5), torch.tensor([0., .75, 1., 1.]))

### 给每张图独立抽一次参数

配方用上限控制变化范围。例如 `shift=2` 时，水平与垂直位移各自在 −2 到 +2 像素间抽取；`light=.15` 时，亮度倍率在 0.85–1.15 之间。旋转绕图心进行，非整数坐标用邻近四格插值，移入的边缘用最近的边界像素填充。对 28 × 28 小图，平移 2 格已占宽度的约 7%。

**动手配方**　修改下格中的 `recipe`。先对同一张照片抽出五个视图，再换一个字母做目检。判断时同时看手指、指缝和照片边界。旋转幅度从 8° 增到 30° 时，哪些类最值得再看一眼？

In [ ]:
recipe = dict(angle=8., shift=2., light=.15)

def random_views(images, rng, angle, shift, light):
    # 每个参数各抽一列，每张图拿到自己的一组值。
    draws = 2 * torch.rand((4, len(images)), generator=rng) - 1
    rotation, right, down, exposure = draws
    return lab.transform_images(images, degrees=rotation * angle,
                                dx=right * shift, dy=down * shift,
                                gain=1 + exposure * light)

letter = "V"
index = torch.where(y_train == lab.LETTERS.index(letter))[0][0]
original = x_train[index:index+1]
copies = original.repeat(5, 1, 1, 1)
views = random_views(copies, torch.Generator().manual_seed(12), **recipe)
fig, axes = plt.subplots(1, 6, figsize=(9, 2.1))
for i, (ax, image) in enumerate(zip(axes, torch.cat((original, views)))):
    ax.imshow(image[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title("Original" if i == 0 else f"View {i}")
    ax.axis("off")
plt.show()
unchanged = random_views(copies, torch.Generator().manual_seed(12), angle=0, shift=0, light=0)
torch.testing.assert_close(unchanged, copies, atol=3e-6, rtol=0)
assert views.shape == copies.shape and 0 <= views.min() <= views.max() <= 1

## 3 · 先分数据，再在线增强

| 时刻 | 图片走哪条路 | 用途 |
| --- | --- | --- |
| 抽样 | 从已有训练集每类取 80 张，共 1,920 张 | 三组共享相同的已标注原图 |
| 每次训练 | 原图 → 抽变换 → 生成本批视图 → 更新参数 | 下一轮再看到这张原图时重新抽参数 |
| 每轮验证 | 原验证图 → 预测 | 读学习曲线 |
| 固定扰动评估 | 对同一批留出图片统一右移 2 格、亮度乘 0.85 → 预测 | 单独测一次指定拍摄变化 |

如果先做五个视图，再打乱分成训练和验证，模型可能在验证时见到训练图的近似副本。本实验沿用已有划分，只在训练批次里抽随机增强。一个 epoch 依旧看 1,920 个样本，各配方更新次数相同。原数据本身含增强变体，因此这里测量的是**额外在线增强**的作用。

**实验前押注**　A 使用原图；B 加平移和旋转；C 在 B 上再加亮度变化。预测哪组在原图验证集上最好，哪组更能应对固定扰动。B 与 C 的差能回答哪个问题？

训练图每轮刷新。Train 是本轮各批实际训练图的平均交叉熵，Validation 是轮末固定权重后在验证原图上的平均交叉熵。先观察两线是否一起下降，再结合验证准确率解释差距。 增强组的 Train 包含在线生成的视图。

In [ ]:
train = lab.few_per_class(signs["train"], 80)
policies = {
    "A / Original": None,
    "B / Geometry": dict(recipe, light=0.),
    "C / Geometry + light": recipe.copy(),
}
epochs = 8
models, histories = {}, {}
for name, policy in policies.items():
    torch.manual_seed(42)
    model = lab.SmallCNN(len(lab.LETTERS)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=.002)
    transform = None
    if policy is not None:
        def transform(images, rng, config=policy):
            return random_views(images, rng, **config)
    histories[name] = lab.fit(model, train, signs["valid"], optimizer, epochs, transform=transform, title=name)
    models[name] = model
    print(name, "完成训练")

## 4 · 在两种拍摄条件下判读结果

下格左侧画每轮原图验证准确率。右侧将同一批验证样本的原图与固定扰动视图放在一起评分。先给一个部署情境，比如“相机支架和灯光固定”或“经常在较暗教室拍摄”，再依据相应的柱子选配方。固定扰动只代表这一种条件。

In [ ]:
def shifted_camera(split):
    images, labels = split
    return lab.transform_images(images, dx=2, gain=.85), labels

valid_shifted = shifted_camera(signs["valid"])
fig, (curve, bars) = plt.subplots(1, 2, figsize=(9, 3.5))
for i, (name, history) in enumerate(histories.items()):
    curve.plot(range(1, epochs+1), history.valid_accuracy, ".-", label=name[0])
    bars.bar(i-.18, history.valid_accuracy[-1], width=.36, color="#247970", label="Original" if i==0 else None)
    bars.bar(i+.18, lab.accuracy(models[name], valid_shifted), width=.36,
             color="#dc9453", label="Shift + dim" if i==0 else None)
curve.set(xlabel="Epoch", ylabel="Validation accuracy", ylim=(0, 1))
bars.set(xticks=range(3), xticklabels=["A", "B", "C"], ylabel="Validation accuracy", ylim=(0, 1))
curve.legend()
bars.legend()
fig.tight_layout()
plt.show()

**锁定配方再打开测试集**　把本轮选择及理由写在下一格的注释中。用相同的测试原图和固定扰动各考一次，按列比较 A、B、C，再按行看改变拍摄条件带来的差距。差值单位是百分点。若继续改配方，请依据验证结果；反复针对同一份测试成绩做选择会使最终报告过于乐观。

下格用每轮保存的权重在测试原图上回算交叉熵。两幅图的纵轴刻度相同，可以逐轮对照训练与测试损失。测试曲线用于最后复盘；选择轮数和超参数请依据验证集。

In [ ]:
# 我的应用情境和配方选择：______
test_shifted = shifted_camera(signs["test"])
table = ["| 配方 | 原图测试准确率 | 固定扰动测试准确率 | 扰动减原图（百分点） |",
         "| --- | ---: | ---: | ---: |"]
for name, model in models.items():
    clean = lab.accuracy(model, signs["test"])
    shifted = lab.accuracy(model, test_shifted)
    table.append(f"| {name} | {clean:.2%} | {shifted:.2%} | {100*(shifted-clean):+.2f} |")
display(Markdown("\n".join(table)))

compared = {name: (model, histories[name]) for name, model in models.items()}
lab.plot_final_losses(compared, signs["test"], "Sign MNIST")


### 提交你的配方

1. 给出角度、位移和亮度范围，点名一个字母并描述使你接受或放弃某个强度的视觉线索。
2. 在固定扰动测试列中，算出 B 减 A、C 减 B 各多少个百分点。哪一个差值最接近你的开场猜想？找出从原图到扰动图成绩下降最少的一组，再检查它在扰动图上的准确率排名。只看下降幅度会漏掉什么？
3. 如果 C 的原图成绩下降，而固定扰动成绩上升，你会为哪种拍摄环境保留它？还有哪种真实变化需要另收样本检验？

**选学**　只把 `shift` 从 2 改成 4，重新生成视图，用同一预算训练，再检查验证图表。用边界附近的一张训练照片解释观察结果。